In [ ]:
import pandas as pd
import os
from upsetplot import UpSet
from upsetplot import from_contents, from_memberships
from pandas import ExcelWriter
import openpyxl as px

In [ ]:
def add_autofilter(fn):
    # based on https://stackoverflow.com/questions/51566349/openpyxl-how-to-add-filters-to-all-columns
    wb = px.load_workbook(fn)
    for ws in wb.worksheets:
        ws.auto_filter.ref = ws.dimensions
    wb.save(fn)

In [ ]:
reg = pd.read_csv('regulations-2024-08-06.csv')
# deduplicate
reg = reg.drop_duplicates()

# take only the sigma factors
reg = reg.loc[reg['mode'].isin(['sigma factor', 'Sigma factor']),:]
reg['regulator name'].unique()

In [ ]:
dd = 'DESeq_all'

od = 'assign_sigma'
os.makedirs(od, exist_ok=True)

joined_results_file = os.path.join(od, 'DE_with_sigma.xlsx')
sigma_counts = os.path.join(od, 'sigma_counts.xlsx')

with ExcelWriter(joined_results_file) as writer, ExcelWriter(sigma_counts) as writer2:
    ard = reg.groupby('regulator name').apply(len)
    ard.reset_index().to_excel(writer2, sheet_name='all_regs_dedup')
    
    for fn in [i for i in os.listdir(dd) if i.endswith('.csv')]:
        de = pd.read_csv(os.path.join(dd, fn), sep=',').rename(columns={'Unnamed: 0': 'locus_id'})
        de['lfccat'] = de.log2FoldChange.apply(lambda x: 'UP' if x>0 else 'DOWN')

        # take only LFC>=+-2 and padj < 0.05
        de = de.loc[(de.log2FoldChange.abs()>=2) & (de.padj < 0.05),:]

        # merge the tables
        m = pd.merge(de, reg, left_on='locus_id', right_on='gene locus', how='left')
        m.loc[m['regulator name'].isna(), 'regulator name'] = 'N/A'

        de2 = de.set_index(de.locus_id)
        de2['regulators'] = m.groupby('locus_id').apply(lambda x: tuple(sorted(x['regulator name'])))
        de2.reset_index(drop=True, inplace=True)
        updata = from_memberships(de2['regulators'], de2)

        # plot
        # fig = plt.Figure()
        # upset = UpSet(
        #     updata,
        #     intersection_plot_elements=0,  # disable the default bar chart
        #     sort_by='cardinality',
        #     show_counts=True,
        #     facecolor='#291528'
        #     #facecolor='#227C9D'
        # )
        # upset.add_stacked_bars(by="lfccat", colors={'DOWN':'#FE6D73', 'UP':'#227C9D'})
        # axdict = upset.plot(fig)
        # t = axdict['totals'].get_position()
        # s = axdict['extra0'].get_position()
        # axdict['extra0'].legend(
        #     bbox_to_anchor=(t.x0, s.y1), loc="upper left",
        #     bbox_transform=fig.transFigure
        # )
        # fig.suptitle(fn.split('.')[0][14:] + ' LFC> +-2')
        # fig.savefig(os.path.join(od, fn[:-3] + 'upset.pdf'), format='pdf', bbox_inches='tight')

        # updated requirements for exported lists
        # - for each group (in upset plot) print row to a table
        export_data = updata.copy()
        export_data['regulators'] = updata['regulators'].apply(lambda x: ', '.join(x))
        export_data = export_data.reset_index()
        col = list(export_data.columns)
        export_data = export_data.loc[:,list(de.columns) + ['regulators']]
        export_data = export_data.sort_values('regulators', ascending=False)
        # xlexport = os.path.join(od, f'{fn[:-4]}_export.xlsx')
        # export_data.to_excel(xlexport, index=False)

        # add_autofilter(xlexport)
        
        # merged data
        export_data.to_excel(writer, index=False, sheet_name=fn[len('DESeq2results_'):-4])
        
        a2 = updata['regulators'].reset_index().drop(columns=['regulators']).apply(sum)
        a2.to_excel(writer2, sheet_name=fn[len('DESeq2results_'):-4])
        
        b = (a2/ard).dropna()
        b = pd.concat([b, (1-b),], axis=1)*100
        bf = b.sort_values(0, ascending=False).plot(kind='bar', stacked=True, color=['black', 'gray'], legend=False)
        bf.figure.savefig(od + '/' + fn[len('DESeq2results_'):-4] + '_all.svg')
        
        bf = b.loc[[i for i in b.index if i.startswith('sig')],:].sort_values(0, ascending=False).plot(kind='bar', stacked=True, color=['black', 'gray'], legend=False)
        bf.figure.savefig(od + '/' + fn[len('DESeq2results_'):-4] + '_sig.svg')

add_autofilter(joined_results_file)

In [ ]:
export_data.sort_values('baseMean',ascending=False).head()

In [ ]:
de2.sort_values('baseMean',ascending=False).head()